In [7]:
import time
import numpy as np
import pandas as pd
from IPython.display import display


def show_cases(series: pd.Series, cases: list[tuple[str, dict]], title: str) -> None:
    out = pd.DataFrame({"original": series})
    for label, kwargs in cases:
        out[label] = series.interpolate(method="linear", **kwargs)
    print(f"\n=== {title} ===")
    display(out)


def strict_limit_interpolate_loop(
    series: pd.Series,
    limit: int,
    *,
    method: str = "linear",
    limit_direction: str = "both",
    limit_area: str | None = None,
) -> pd.Series:
    if limit < 1:
        raise ValueError("limit must be >= 1")

    interpolated = series.interpolate(
        method=method,
        limit_direction=limit_direction,
        limit_area=limit_area,
    )
    result = interpolated.copy()

    is_nan = series.isna()
    groups = is_nan.ne(is_nan.shift(fill_value=False)).cumsum()

    for _, idx in series.groupby(groups).groups.items():
        idx = pd.Index(idx)
        if not is_nan.loc[idx].all():
            continue
        if len(idx) > limit:
            result.loc[idx] = np.nan

    return result


def strict_limit_interpolate_vectorized(
    series: pd.Series,
    limit: int,
    *,
    method: str = "linear",
    limit_direction: str = "both",
    limit_area: str | None = None,
) -> pd.Series:
    if limit < 1:
        raise ValueError("limit must be >= 1")

    interpolated = series.interpolate(
        method=method,
        limit_direction=limit_direction,
        limit_area=limit_area,
    )

    is_nan = series.isna()
    run_id = is_nan.ne(is_nan.shift(fill_value=False)).cumsum()
    run_len = is_nan.groupby(run_id).transform("sum")

    return interpolated.mask(is_nan & (run_len > limit))


# Pattern includes:
# - leading NaNs
# - an internal run of NaNs
# - trailing NaNs
s = pd.Series(
    [np.nan, np.nan, 1.0, np.nan, np.nan, np.nan, 5.0, np.nan, 9.0, np.nan, np.nan],
    name="x",
)

print("Input series:")
display(pd.DataFrame({"original": s}))

# 1) Test `limit`
limit_cases = [
    ("limit=None (default)", {}),
    ("limit=1", {"limit": 1}),
    ("limit=2", {"limit": 2}),
    ("limit=10", {"limit": 10}),
]
show_cases(s, limit_cases, "Effect of limit")

# 1b) Strict behavior you described
strict_limit_compare = pd.DataFrame({
    "original": s,
    "pandas limit=2": s.interpolate(method="linear", limit=2),
    "strict limit=2 (loop)": strict_limit_interpolate_loop(s, limit=2),
    "strict limit=2 (vectorized)": strict_limit_interpolate_vectorized(s, limit=2),
})
print("\n=== Strict limit behavior (all-or-nothing per NaN run) ===")
display(strict_limit_compare)

# 2) Test `limit_direction`
# Use limit=2 so direction changes what gets filled in the 3-NaN internal gap.
limit_direction_cases = [
    ("limit=2, direction='forward'", {"limit": 2, "limit_direction": "forward"}),
    ("limit=2, direction='backward'", {"limit": 2, "limit_direction": "backward"}),
    ("limit=2, direction='both'", {"limit": 2, "limit_direction": "both"}),
]
show_cases(s, limit_direction_cases, "Effect of limit_direction")

# 3) Test `limit_area`
# Use a large limit + direction='both' so area restriction is clear.
limit_area_cases = [
    ("limit=10, area='inside'", {"limit": 10, "limit_direction": "both", "limit_area": "inside"}),
    ("limit=10, area='outside'", {"limit": 10, "limit_direction": "both", "limit_area": "outside"}),
    ("limit=10, area=None", {"limit": 10, "limit_direction": "both"}),
]
show_cases(s, limit_area_cases, "Effect of limit_area")


# 4) Speed benchmark: loop vs vectorized strict interpolation

def make_benchmark_series(n: int, nan_rate: float = 0.35, seed: int = 0) -> pd.Series:
    rng = np.random.default_rng(seed)
    vals = pd.Series(rng.normal(size=n), name="x")

    mask = rng.random(n) < nan_rate

    # Inject additional long NaN runs so strict-limit logic does real work.
    long_run_count = max(1, n // 50000)
    starts = rng.integers(low=10, high=max(11, n - 40), size=long_run_count)
    for st in starts:
        end = min(n, st + 20)
        mask[st:end] = True

    mask[0] = False
    mask[-1] = False
    vals.loc[mask] = np.nan
    return vals


def time_call(fn, series: pd.Series, repeats: int = 3, warmup: int = 1, **kwargs) -> float:
    for _ in range(warmup):
        fn(series, **kwargs)

    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(series, **kwargs)
        times.append(time.perf_counter() - t0)

    return min(times)


benchmark_sizes = [10_000, 100_000, 300_000]
benchmark_limit = 3
benchmark_rows = []

for n in benchmark_sizes:
    x = make_benchmark_series(n=n, seed=42)

    # Correctness check once per size.
    loop_out = strict_limit_interpolate_loop(x, limit=benchmark_limit)
    vec_out = strict_limit_interpolate_vectorized(x, limit=benchmark_limit)
    same = loop_out.equals(vec_out)

    t_loop = time_call(
        strict_limit_interpolate_loop,
        x,
        limit=benchmark_limit,
        repeats=3,
        warmup=1,
    )
    t_vec = time_call(
        strict_limit_interpolate_vectorized,
        x,
        limit=benchmark_limit,
        repeats=3,
        warmup=1,
    )

    benchmark_rows.append(
        {
            "n": n,
            "loop_seconds": t_loop,
            "vectorized_seconds": t_vec,
            "speedup_x (loop/vec)": t_loop / t_vec if t_vec > 0 else np.nan,
            "outputs_match": same,
        }
    )

benchmark_df = pd.DataFrame(benchmark_rows)
print("\n=== Benchmark: strict interpolation loop vs vectorized ===")
display(benchmark_df)

Input series:


,original
0,NaN
1,NaN
2,1.0
3,NaN
4,NaN
5,NaN
6,5.0
7,NaN
8,9.0
9,NaN



=== Effect of limit ===


,original,limit=None (default),limit=1,limit=2,limit=10
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN
2,1.0,1.0,1.0,1.0,1.0
3,NaN,2.0,2.0,2.0,2.0
4,NaN,3.0,NaN,3.0,3.0
5,NaN,4.0,NaN,NaN,4.0
6,5.0,5.0,5.0,5.0,5.0
7,NaN,7.0,7.0,7.0,7.0
8,9.0,9.0,9.0,9.0,9.0
9,NaN,9.0,9.0,9.0,9.0



=== Strict limit behavior (all-or-nothing per NaN run) ===


...

,original,"limit=2, direction='forward'","limit=2, direction='backward'","limit=2, direction='both'"
0,NaN,NaN,1.0,1.0
1,NaN,NaN,1.0,1.0
2,1.0,1.0,1.0,1.0
3,NaN,2.0,NaN,2.0
4,NaN,3.0,3.0,3.0
5,NaN,NaN,4.0,4.0
6,5.0,5.0,5.0,5.0
7,NaN,7.0,7.0,7.0
8,9.0,9.0,9.0,9.0
9,NaN,9.0,NaN,9.0



=== Effect of limit_area ===


,original,"limit=10, area='inside'","limit=10, area='outside'","limit=10, area=None"
0,NaN,NaN,1.0,1.0
1,NaN,NaN,1.0,1.0
2,1.0,1.0,1.0,1.0
3,NaN,2.0,NaN,2.0
4,NaN,3.0,NaN,3.0
5,NaN,4.0,NaN,4.0
6,5.0,5.0,5.0,5.0
7,NaN,7.0,NaN,7.0
8,9.0,9.0,9.0,9.0
9,NaN,NaN,9.0,9.0



=== Benchmark: strict interpolation loop vs vectorized ===


,n,loop_seconds,vectorized_seconds,speedup_x (loop/vec),outputs_match
0,10000,0.369107,0.002093,176.357296,True
1,100000,3.767482,0.016090,234.155739,True
2,300000,11.279931,0.050228,224.576631,True
